# Scalable MatMul-free Language Modeling

Zhu, Rui-Jie, Yu Zhang, Ethan Sifferman, et al. "Scalable MatMul-Free Language Modeling." arXiv:2406.02528. Preprint, arXiv, June 18, 2024. https://doi.org/10.48550/arXiv.2406.02528

In [ ]:
# \!pip install torch datasets matplotlib

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from dataclasses import dataclass

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Eszköz: {device}")

In [ ]:
torch.manual_seed(42)

In [ ]:
@dataclass
class ModelConfig:
    vocab_size: int   = 256
    d_model:    int   = 128
    n_layers:   int   = 4
    n_heads:    int   = 4
    seq_len:    int   = 256
    dropout:    float = 0.1

config = ModelConfig()

## BitLinear – Mátrixszorzás nélküli lineáris réteg

Egy hagyományos `nn.Linear` réteg a következő műveletet végzi:

$$\mathbf{y} = \mathbf{x} \mathbf{W}^T$$

ahol $\mathbf{x} \in \mathbb{R}^{d_{in}}$ a bemenet, $\mathbf{W} \in \mathbb{R}^{d_{out} \times d_{in}}$ a súlymátrix. Ez egy **lebegőpontos mátrixszorzás**, amelynek bonyolultsága $O(d_{in} \cdot d_{out})$. Egy nagy modellben ezek teszik ki a számítás ~95%-át.

A BitLinear ezt váltja ki úgy, hogy a súlyokat **ternáris értékekre** ($\{-1, 0, +1\}$) kvantálja, az aktivációkat pedig **8-bites egészekre** – a szorzások így összeadásokká alakíthatók.

### RMSNorm – normalizáció

A kvantálás előtt az értékeket normalizálni kell – különben a kvantálás pontatlan lenne.

Az **RMSNorm** (Root Mean Square Normalization) képlete:

$$\text{RMSNorm}(\mathbf{x}) = \frac{\mathbf{x}}{\text{RMS}(\mathbf{x})} \cdot \boldsymbol{\gamma}$$

ahol:

$$\text{RMS}(\mathbf{x}) = \sqrt{\frac{1}{d} \sum_{i=1}^{d} x_i^2 + \varepsilon}$$

A $\boldsymbol{\gamma} \in \mathbb{R}^d$ **tanulható skálázási vektor** (kezdetben csupa 1), az $\varepsilon$ kis szám a numerikus stabilitáshoz.

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-8):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        rms_inv = x.pow(2).mean(dim=-1, keepdim=True).add(self.eps).rsqrt()
        return x * rms_inv * self.weight

Ellenőrzés.

In [ ]:
x_test = torch.randn(2, 10, config.d_model)
norm = RMSNorm(config.d_model)
x_normed = norm(x_test)

print(f"Bemenet alakja:  {x_test.shape}")
print(f"Kimenet alakja:  {x_normed.shape}")
print(f"Bemenet  átlag={x_test.mean():.3f}  std={x_test.std():.3f}")
print(f"Kimenet  átlag={x_normed.mean():.3f}  std={x_normed.std():.3f}")
print("A kimenet értékei normalizálódtak (std közel 1-hez)")

### Súlykvantálás – ternáris értékekre ($\{-1, 0, +1\}$)

**Lépések:**

**1.** Skálafaktor: az abszolút értékek átlaga:
$$\alpha = \frac{1}{nm} \sum_{ij} |W_{ij}|$$

**2.** Normálás és kerekítés:
$$\tilde{W}_{ij} = \text{round}\left(\frac{W_{ij}}{\alpha}\right)$$

**3.** Szorítás $[-1, 1]$-re:
$$W^{\text{quant}}_{ij} = \text{clamp}(\tilde{W}_{ij},\ -1,\ 1)$$

**4.** Visszaskálázás (hogy a numerikus skála megmaradjon):

$$W^{\text{final}}_{ij} = W^{\text{quant}}_{ij} \cdot \alpha$$

In [ ]:
def weight_quant(w: torch.Tensor) -> torch.Tensor:
    scale = 1.0 / w.abs().mean().clamp_(min=1e-5)
    return (w * scale).round().clamp_(-1, 1) / scale

In [ ]:
w_ex = torch.randn(64, 64)
w_q  = weight_quant(w_ex)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(w_ex.flatten().numpy(), bins=50, color='lightblue', alpha=0.8)
axes[0].set_title('Súlyok kvantálás ELŐTT\n(folytonos lebegőpontos értékek)')
axes[0].set_xlabel('Súlyérték')
axes[0].set_ylabel('Darabszám')

axes[1].hist(w_q.flatten().detach().numpy(), bins=50, color='red', alpha=0.8)
axes[1].set_title('Súlyok kvantálás UTÁN\n(csak -1, 0, +1 értékek)')
axes[1].set_xlabel('Súlyérték')

plt.tight_layout()
plt.show()

unique_vals = sorted(w_q.unique().tolist())
print(f"Egyedi értékek kvantálás után: {[round(v,3) for v in unique_vals]}")
print(f"Csak 3 különböző érték van (ternáris kvantálás)!")

### Aktivációkvantálás – 8 bitre

Az aktivációkat, azaz a réteg bemeneti vektorait a rendszer token-szinten kvantálja 8-bites egészekre, a $[−128,127]$ tartományba.

A token-szintű kvantálás azt jelenti, hogy minden pozícióhoz külön skálafaktor kerül kiszámításra, így minden token kihasználja a saját dinamikatartományát.

A dinamikatartomány azt jelzi, hogy egy token értékei milyen széles tartományban szóródnak. Azért fontos, hogy minden token a saját tartományához igazodjon: így a 8-bites felbontás nem „vész el" olyan értékkészletre, amely az adott tokennél nem fordul elő.

A képletekben $t$ a token pozícióját, $j$ pedig az adott token feature-vektorának
dimenzióindexét jelöli.

**Lépések:**

**1.** Token-szintű skálafaktor ($t$-edik tokenre):
$$s_t = \frac{127}{\max_j |x_{t,j}|}$$

**2.** Skálázás, kerekítés, szorítás:
$$x^{\text{int8}}_{t,j} = \text{clamp}\left(\text{round}(x_{t,j} \cdot s_t),\ -128,\ 127\right)$$

**3.** Visszaskálázás:
$$x^{\text{final}}_{t,j} = \frac{x^{\text{int8}}_{t,j}}{s_t}$$

In [ ]:
def activation_quant(x: torch.Tensor) -> torch.Tensor:
    scale = 127.0 / x.abs().max(dim=-1, keepdim=True).values.clamp_(min=1e-5)
    return (x * scale).round().clamp_(-128, 127) / scale

In [ ]:
x_act = torch.randn(4, 10, 32)
x_qact = activation_quant(x_act)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(x_act.flatten().numpy(), bins=60, color='steelblue', alpha=0.8)
axes[0].set_title('Aktiváció kvantálás ELŐTT')
axes[0].set_xlabel('Érték')
axes[1].hist(x_qact.flatten().detach().numpy(), bins=60, color='tomato', alpha=0.8)
axes[1].set_title('Aktiváció kvantálás UTÁN\n(diszkrét 8-bites értékek)')
axes[1].set_xlabel('Érték')
plt.tight_layout()
plt.show()

### Straight-Through Estimator (STE)

A probléma: A matematika szabályai szerint a valódi gradienst így kéne kiszámolni:$$\frac{\partial \mathcal{L}}{\partial x} = \frac{\partial \mathcal{L}}{\partial x^{\text{quant}}} \cdot \frac{\partial x^{\text{quant}}}{\partial x}$$


Mivel azonban a kerekítés egy lépcsős függvény, a deriváltja ($\frac{\partial x^{\text{quant}}}{\partial x}$) majdnem mindenhol nulla. Ha nullával történik a szorzás, a gradiens eltűnik, és a tanulás leáll.

A megoldás:  Előrefelé a kvantálás megtörténik, hátrafelé viszont úgy viselkedik a rendszer, mintha a kvantálás meg sem történt volna.

$$x^{\text{quant}} = x + \underbrace{(\text{round}(x) - x)}_{\text{.detach()}}$$

Forward pass: A .detach() az értékeket nem bántja, így az eredmény egyszerűen $\text{round}(x)$ lesz. A modell a kvantált értékkel számol.

Backward pass: A .detach() levágja a kerekítési műveletet a PyTorch számítási gráfjáról. A gradiens így sértetlenül "átfolyik" az önálló $x$ tagon, kikerülve a halott gradienseket.

Matematikailag ez egy közelítés:

$$\frac{\partial \mathcal{L}}{\partial x} \approx \frac{\partial \mathcal{L}}{\partial x^{\text{quant}}}$$

Ez a képlet azt mutatja, hogy a láncszabály kiszámításakor szándékosan ignoráljuk a kerekítés deriváltját. Így a gradiens akadálytalanul eljut a háttérben lévő lebegőpontos súlyokhoz, amelyek lassan addig frissülnek, amíg át nem billennek a következő kvantált állapotba pl. 0-ból 1-be.

In [ ]:
x_demo = torch.tensor([0.7, -0.3, 1.2, -0.8], requires_grad=True)

x_demo.grad = None
x_rounded = x_demo.round()
x_rounded.sum().backward()
print(f"Gradiens STE NÉLKÜL: {x_demo.grad}")
print("Csupa nulla! A round() gradiensei nullák, a modell nem tanulna.")
print()

x_demo.grad = None
x_ste = x_demo + (x_demo.round() - x_demo).detach()  # ← STE trükk
x_ste.sum().backward()
print(f"Gradiens STE-VEL:    {x_demo.grad}")
print("Csupa 1.0! A gradiens átment, a modell tanulni tud.")
print()
print(f"Forward értékek STE-vel: {x_ste.data}")
print(f"Eredeti értékek:         {x_demo.data}")
print("Forward passban kerekített értékek, backward passban valódi gradiens!")

### BitLinear – az összes elem összerakva



A teljes forward pass:

$$y = F_{\text{linear}}\left(\hat{x}^{\text{quant}},\ \hat{W}^{\text{quant}}\right)$$

ahol:
- $\hat{x} = \text{RMSNorm}(x)$ – normalizált bemenet
- $\hat{x}^{\text{quant}} = \hat{x} + (\text{activation_quant}(\hat{x}) - \hat{x})\text{.detach()}$ – STE-kvantált aktiváció
- $\hat{W}^{\text{quant}} = W + (\text{weight_quant}(W) - W)\text{.detach()}$ – STE-kvantált súlyok

In [ ]:
class BitLinear(nn.Linear):
    def __init__(self, in_features: int, out_features: int, bias: bool = False):
        super().__init__(in_features, out_features, bias=bias)
        self.norm = RMSNorm(in_features, eps=1e-8)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x_norm = self.norm(x)
        x_quant = x_norm + (activation_quant(x_norm) - x_norm).detach()
        w = self.weight
        w_quant = w + (weight_quant(w) - w).detach()
        return F.linear(x_quant, w_quant, self.bias)

In [ ]:
# Ellenőrzés: BitLinear vs nn.Linear
batch = 2
seq = 10
d_in = d_out = config.d_model

x_test = torch.randn(batch, seq, d_in)

# Hagyományos lineáris réteg
linear    = nn.Linear(d_in, d_out, bias=False)

# BitLinear ugyanolyan súlyokkal
bitlinear = BitLinear(d_in, d_out, bias=False)
bitlinear.weight.data = linear.weight.data.clone()

y_lin = linear(x_test)
y_bit = bitlinear(x_test)

print(f"nn.Linear  kimenet alakja: {y_lin.shape}")
print(f"BitLinear  kimenet alakja: {y_bit.shape}")
print()
print(f"nn.Linear  első értékek: {y_lin[0, 0, :6].detach().numpy().round(3)}")
print(f"BitLinear  első értékek: {y_bit[0, 0, :6].detach().numpy().round(3)}")
print()

# Memóriamegtakarítás szemléltetése
total_weights = d_in * d_out
print(f"Összes súly száma: {total_weights:,}")
print(f"nn.Linear  memória: {total_weights * 4:,} bájt  (float32, 4 bájt/érték)")
#4bajt miért?
print(f"BitLinear  memória: {total_weights * 1.58 / 8:.0f} bájt  (ternáris, 1.58 bit/érték)")
print(f"Megtakarítás: ~{4 / (1.58/8):.0f}x")

## MLGRU – Lineáris rekurrencia az önfigyelem helyett

 A hagyományos önfigyelem (self-attention) legnagyobb szűk keresztmetszete a $\mathbf{Q}\mathbf{K}^T$ mátrixszorzat, amelynek kiszámítása $O(n^2 d)$ bonyolultságú (ahol $n$ a szekvencia hossza).

 Bár a BitLinear rétegekkel a kezdeti vetítések kvantálhatók, a dinamikus $\mathbf{Q}\mathbf{K}^T$ mátrixszorzat továbbra is megmaradna, ami lehetetlenné tenné a MatMul-mentes működést.

 Ennek kiküszöbölésére a modell az önfigyelmet egy kapuzott lineáris rekurrens egységgel (MLGRU) váltja ki. Ez a mechanizmus a szekvencia feldolgozásához kizárólag elem-szintű szorzásokat ($\odot$) és összeadásokat használ, így a mátrixszorzás maradéktalanul eltűnik a folyamatból.

### A rekurrencia képlete

Legyen $\mathbf{x}_t \in \mathbb{R}^{d}$ a $t$-edik token rejtett állapota a bemeneti szekvenciából. Az MLGRU időlépésenként frissíti a **rejtett állapotot** $\mathbf{h}_t \in \mathbb{R}^{d}$:

$$\mathbf{h}_t = \mathbf{f}_t \odot \mathbf{h}_{t-1} + \mathbf{i}_t \odot \mathbf{x}_t$$

ahol:

- $\mathbf{h}_t$ – a **rejtett állapot** a $t$-edik pozícióban: tömöríti az eddigi szekvencia „emlékezetét”
- $\mathbf{f}_t \in (0, 1)^d$ – a **felejtési kapu** *(forget gate)*: meghatározza, hogy az előző állapotból mennyi „marad meg”. Ha $f_{t,i}$ közel 1-hez → az $i$-edik dimenzió emlékezik a múltra. Ha közel 0-hoz → elfelejti.
- $\mathbf{i}_t$ – a **bemeneti kapu** *(input gate)*: meghatározza az aktuális bemenet átengedésének mértékét.
- $\odot$ – **elem-szintű szorzás** (Hadamard-szorzat): nem mátrixszorzás, hanem pozíciónkénti szorzás!

A kapuk értékeit a bemenetből számítják BitLinear rétegekkel:

$$\mathbf{f}_t = \sigma\left(\text{BitLinear}_f(\mathbf{x}_t)\right)$$
$$\mathbf{i}_t = \text{SwiGLU}\left(\text{BitLinear}_i(\mathbf{x}_t),\ 1 - \mathbf{f}_t\right)$$

ahol $\sigma$ a szigmoid függvény. (értéke $(0, 1)$ között van)

### SwiGLU – az aktivációs függvény



A SwiGLU (Swish-Gated Linear Unit) egy kapuzott aktivációs függvény:

$$\text{SwiGLU}(\mathbf{a}, \mathbf{b}) = \mathbf{a} \odot \text{SiLU}(\mathbf{b})$$

ahol a SiLU (Sigmoid Linear Unit):

$$\text{SiLU}(x) = x \cdot \sigma(x) = \frac{x}{1 + e^{-x}}$$

Ez a függvény simán szabályozza, hogy melyik információ „áramlik át" – a $\mathbf{b}$ értéke kapuként működik, $\mathbf{a}$ az átbocsátott jel.

In [ ]:
def silu(x: torch.Tensor) -> torch.Tensor:
    return x * torch.sigmoid(x)

def swiglu(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    return a * silu(b)

In [ ]:
x_vis = torch.linspace(-4, 4, 200)

plt.figure(figsize=(8, 4))
plt.plot(x_vis.numpy(), silu(x_vis).numpy(), label='SiLU(x)', color='blue', linewidth=2)
plt.plot(x_vis.numpy(), torch.relu(x_vis).numpy(), label='ReLU(x)', color='red', linewidth=2)
plt.axhline(0, color='black', linewidth=0.5)
plt.axvline(0, color='black', linewidth=0.5)
plt.title('SiLU és ReLU aktivációs függvények')
plt.xlabel('x')
plt.ylabel('Aktiváció értéke')
plt.legend()
plt.grid(alpha=0.5)
plt.show()
print("A SiLU simán negatív értékeket is átenged, ami jobb gradienst biztosít")

### MLGRU blokk implementációja

A szekvencia **idősorban kerül feldolgozásra**: minden tokenre kiszámításra kerülnek a kapuk, majd frissül a rejtett állapot.

A blokk felépítése:

1. **BitLinear vetítések** → $\mathbf{f}_t$ (felejtési kapu) és $\mathbf{i}_t$ (bemeneti érték) előállítása
2. **Rekurrencia** → $\mathbf{h}_t = \mathbf{f}_t \odot \mathbf{h}_{t-1} + \mathbf{i}_t \odot \mathbf{x}_t$
3. **Kimeneti kapu** → egy BitLinear vetítés a kimeneti értékre
4. **Kimeneti normalizáció** → RMSNorm a kimenet stabilizálásához

In [ ]:
class MLGRULayer(nn.Module):
    def __init__(self, d_model: int, n_heads: int):
        super().__init__()

        self.d_model = d_model
        self.n_heads = n_heads

        self.head_dim = d_model // n_heads

        # Felejtési kapu vetítése: x → f
        self.f_proj = BitLinear(d_model, d_model, bias=False)

        # Bemenet vetítése: x → i
        self.i_proj = BitLinear(d_model, d_model, bias=False)

        # Kimeneti kapu vetítése: x → g
        self.g_proj = BitLinear(d_model, d_model, bias=False)

        # Kimeneti normalizáció: RMSNorm + Swish kapu
        self.out_norm = RMSNorm(d_model)

        # Kimeneti vetítés: a rejtett állapotot visszavetítjük d_model-re
        self.o_proj  = BitLinear(d_model, d_model, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, D = x.shape  # batch, seq_len, d_model

        # Felejtési kapu: sigmoid biztosítja, hogy értéke (0, 1) között legyen
        f = self.f_proj(x).sigmoid()

        # Bemeneti érték: SwiGLU kapuzott aktiváció
        i = swiglu(self.i_proj(x), 1 - f)

        # Rejtett állapot inicializálása nullákkal: (B, n_heads, head_dim)
        h = torch.zeros(B, self.n_heads, self.head_dim, device=x.device, dtype=x.dtype)

        # Átrendezzük (B, T, D) → (B, T, n_heads, head_dim) alakra a fejek kezeléséhez
        # majd (B, n_heads, T, head_dim) alakra (fejek párhuzamos feldolgozásához)
        i_heads = i.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        f_heads = f.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        outputs = []
        for t in range(T):
            h = f_heads[:, :, t, :] * h + i_heads[:, :, t, :]
            outputs.append(h)

        o = torch.stack(outputs, dim=2)
        o = o.transpose(1, 2).contiguous().view(B, T, D)

        # A rejtett állapotot kapuzzuk a bemeneti kapu segítségével
        g = self.g_proj(x)

        # RMSNorm + Swish: normalizáljuk a kimenetet, majd kapuzzuk
        o = self.out_norm(o) * silu(g)

        return self.o_proj(o)

Ellenőrzés.

In [ ]:
# Ellenőrzés: az MLGRU réteg működése
mlgru = MLGRULayer(d_model=config.d_model, n_heads=config.n_heads).to(device)
x_test = torch.randn(2, config.seq_len, config.d_model).to(device)

y_mlgru = mlgru(x_test)

print(f"Bemenet alakja:  {x_test.shape}")
print(f"Kimenet alakja:  {y_mlgru.shape}")
print(f"Az MLGRU réteg megőrzi az alakot: (batch, seq_len, d_model)")
print()

# Paraméterek száma összehasonlítva egy Multi-Head Attention réteggel
mlgru_params = sum(p.numel() for p in mlgru.parameters())
mha = nn.MultiheadAttention(config.d_model, config.n_heads, batch_first=True)
mha_params  = sum(p.numel() for p in mha.parameters())
print(f"MLGRU paraméterei:  {mlgru_params:,}")
print(f"MHA  paraméterei:  {mha_params:,}")
print(f"Hasonló paraméterszám, de MLGRU-ban nincs QK^T mátrixszorzás!")

### BitMLP - Feed-Forward hálózat BitLinear-ral

A transzformer alapú modellek minden blokkja tartalmaz egy Feed-Forward hálózatot (MLP) a sorozatok feldolgozása után. Míg az önfigyelem (vagy az MLGRU) az *időben* keveri az információt a szavak között, addig az MLP a szavak *saját, belső tulajdonságait* (dimenzióit) keveri újra, pozíciónként függetlenül.

A hagyományos transzformerekben ez két egyszerű lineáris vetítésből és egy ReLU aktivációból áll:
$$\text{MLP}(\mathbf{x}) = \mathbf{W}_{\text{down}} \cdot \text{ReLU}(\mathbf{W}_{\text{up}} \mathbf{x})$$

A MatMul-free architektúrában a drága lebegőpontos rétegeket **BitLinear** modulokra cseréljük, a standard ReLU helyett a **SwiGLU** (Swish Gated Linear Unit) mechanizmust használjuk.

A SwiGLU különlegessége, hogy az egyszerű aktiváció helyett *két párhuzamos* lineáris réteget használ: az egyik az adatokat kiterjeszti (`up`), a másik pedig egy „kapuként” (`gate`) működve finoman szabályozza, hogy ebből az információból mi áramolhasson tovább. Az egyenlet a következőképpen alakul:

$$\text{BitMLP}(\mathbf{x}) = \text{BitLinear}_{\text{down}}\left(\text{SwiGLU}\left(\text{BitLinear}_{\text{gate}}(\mathbf{x}),\ \text{BitLinear}_{\text{up}}(\mathbf{x})\right)\right)$$

A belső kiterjesztett dimenzió a hagyományos modellekhez hasonlóan itt is az alapdimenzió többszöröse (általában $4 \times d_{\text{model}}$), ami garantálja a hálózat megfelelő tanulási kapacitását.

In [ ]:
class BitMLP(nn.Module):
    def __init__(self, d_model: int, expansion_factor: int = 4):
        super().__init__()

        # Belső dimenzió: általában 4x a d_model
        # de SwiGLU miatt 2/3-ára csökkentjük, hogy a paraméterszám egyezzen
        d_inner = int(d_model * expansion_factor * 2 / 3)

        # Kerekítjük 64 többszörösére (GPU-hatékonyság)
        d_inner = 64 * ((d_inner + 63) // 64)

        self.gate_proj = BitLinear(d_model, d_inner, bias=False)
        self.up_proj   = BitLinear(d_model, d_inner, bias=False)

        self.down_proj = BitLinear(d_inner, d_model, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.down_proj(swiglu(self.gate_proj(x), self.up_proj(x)))

## MatMulFree blokk

Egy teljes modellblokk az MLGRU réteget és a BitMLP-t kombinálja **reziduális kapcsolatokkal**.

A **reziduális kapcsolat** (skip connection) azt jelenti, hogy a réteg kimenetéhez hozzáadjuk az eredeti bemenetet:

$$\mathbf{x}' = \mathbf{x} + \text{MLGRU}(\text{RMSNorm}(\mathbf{x}))$$
$$\mathbf{x}'' = \mathbf{x}' + \text{BitMLP}(\text{RMSNorm}(\mathbf{x}'))$$

In [ ]:
class MatMulFreeBlock(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()

        self.norm1 = RMSNorm(config.d_model)
        self.norm2 = RMSNorm(config.d_model)

        # A két fő komponens
        self.mlgru = MLGRULayer(config.d_model, config.n_heads)
        self.mlp  = BitMLP(config.d_model)

        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.dropout(self.mlgru(self.norm1(x)))
        x = x + self.dropout(self.mlp(self.norm2(x)))

        return x

## MatMulFree nyelvi modell

A teljes modell:

1. **Token beágyazás** – egész indexekből ($[0, \text{vocab\_size})$) $d$-dimenziós vektorokat csinál
2. **Pozicionális beágyazás** – megmondja a modellnek, hol van az adott token a szekvenciában
3. **N × MatMulFree blokk** – a tényleges feldolgozás
4. **Végső RMSNorm** – a kimenet normalizálása
5. **LM Head (BitLinear)** – visszavetít `vocab_size` méretre → ezek a **logitok** (nyers, nem normalizált valószínűségek)

A logitokból a **softmax** adja meg az egyes tokenek valószínűségét:

$$P(\text{token}_i) = \frac{e^{z_i}}{\sum_j e^{z_j}}$$

In [ ]:
class MatMulFreeLM(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()
        self.config = config

        # Token beágyazás: egész index → d_model dimenziós vektor
        self.tok_emb = nn.Embedding(config.vocab_size, config.d_model)

        # Pozicionális beágyazás: tanulható pozíció-vektorok
        self.pos_emb = nn.Embedding(config.seq_len, config.d_model)

        self.drop = nn.Dropout(config.dropout)

        # N darab MatMulFree blokk egymásra rakva
        self.blocks = nn.ModuleList([MatMulFreeBlock(config) for _ in range(config.n_layers)])

        # Végső normalizáció
        self.norm_final = RMSNorm(config.d_model)

        # LM fej
        self.lm_head = BitLinear(config.d_model, config.vocab_size, bias=False)

        # Súlyinicializálás
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, BitLinear)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx: torch.Tensor, targets: torch.Tensor = None):
        B, T = idx.shape
        assert T <= self.config.seq_len, f"Szekvencia ({T}) meghaladja a max hosszt ({self.config.seq_len})"

        # Token beágyazás
        tok = self.tok_emb(idx)

        # Pozicionális beágyazás
        pos = self.pos_emb(torch.arange(T, device=idx.device))

        # Összeadjuk a token- és pozicionális beágyazásokat (broadcast: (B,T,D) + (T,D))
        x = self.drop(tok + pos)

        # N darab blokkon átfuttatjuk
        for block in self.blocks:
            x = block(x)

        # Végső normalizáció
        x = self.norm_final(x)

        # Logitok: (B, T, d_model) → (B, T, vocab_size)
        logits = self.lm_head(x)

        # Ha van célérték, kiszámítjuk a veszteséget (loss)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, self.config.vocab_size), targets.view(-1))

        return logits, loss

In [ ]:
# Modell inicializálása és paraméterszám ellenőrzése
model = MatMulFreeLM(config).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable   = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Összes paraméter:    {total_params:,}")
print(f"Tanítható paraméter: {trainable:,}")
print()

# Teszt futtatás: véletlen token indexek
x_dummy = torch.randint(0, config.vocab_size, (2, config.seq_len)).to(device)
logits, _ = model(x_dummy)
print(f"Bemenet alakja:  {x_dummy.shape}  (batch=2, seq_len={config.seq_len})")
print(f"Logitok alakja:  {logits.shape}   (batch=2, seq_len={config.seq_len}, vocab={config.vocab_size})")
print("A modell sikeresen lefutott!")

### Adathalmaz és tokenizálás

A legegyszerűbb tokenizálási módszer: **byte-szintű (karakterszintű)** megközelítés kerül alkalmazásra: minden UTF-8 bájt külön tokenként kezelődik.

Az adat **átfedő ablakokra** kerül vágásra: a modell `seq_len` hosszú szövegdarabokat lát, és minden pozícióban megjósolja a következő tokent.

In [ ]:
import os, urllib.request

# TinyShakespeare: ~1MB, Shakespeare osszes muve
if not os.path.exists("input.txt"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt",
        "input.txt"
    )
    print("Letöltve!")

with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

print(f"Szöveg hossza: {len(text):,} karakter")

In [ ]:
# Byte-szintű tokenizálás
# Minden karaktert UTF-8 bájttá alakítunk
data = torch.tensor(list(text.encode('utf-8')), dtype=torch.long)

print(f"Token-sorozat hossza:  {len(data):,}")
print(f"Token értékkészlet:    0-{data.max().item()} (byte értékek)")
print(f"Első 13 token:         {data[:13].tolist()}")
print(f"Visszaalakítva szöveg: {bytes(data[:13].tolist()).decode('utf-8', errors='replace')}")
print()

split = int(len(data) * 0.9)
train_data = data[:split]
val_data   = data[split:]

print(f"Tanítási adatok:    {len(train_data):,} token")
print(f"Validációs adatok:  {len(val_data):,} token")

In [ ]:
def get_batch(data: torch.Tensor, batch_size: int, seq_len: int, device) -> tuple:
    # batch_size darab véletlen kezdőpozíciót választunk
    ix = torch.randint(len(data) - seq_len, (batch_size,))

    # Kiszabunk seq_len hosszú ablakokat minden kezdőpozícióból
    x = torch.stack([data[i      : i + seq_len    ] for i in ix])
    y = torch.stack([data[i + 1  : i + seq_len + 1] for i in ix])

    return x.to(device), y.to(device)

# Ellenőrzés
xb, yb = get_batch(train_data, batch_size=4, seq_len=config.seq_len, device=device)
print(f"Bemenet (x) alakja: {xb.shape}  (batch=4, seq_len={config.seq_len})")
print(f"Célérték (y) alakja: {yb.shape}")
print()
print(f"x első sorának első 10 tokenje:  {xb[0, :10].tolist()}")
print(f"y első sorának első 10 tokenje:  {yb[0, :10].tolist()}")
print()
print("y pontosan egy pozícióval el van tolva x-hez képest!")

## Tanítás

Az egész tanítási loop 4 lépésből áll.

1. **Véletlen mini-batch** kivágása az adatból
2. **Forward pass** – a modell megjósolja a következő tokeneket, kiszámítja a losst
3. **Backward pass** – a PyTorch kiszámítja az összes gradienst (`loss.backward()`)
4. **Súlyfrissítés** – az optimizer (AdamW) frissíti a súlyokat a gradiens alapján

Az **AdamW** optimizer az SGD (gradiens ereszkedés) egy fejlettebb változata, amely:
- Adaptívan állítja a tanulási rátát minden paraméterhez
- Weight decay-t alkalmaz (regularizáció a túltanulás ellen)


A kvantált (ternáris) modellek a kerekítési műveletek miatt a szokásosnál nagyobb tanulási rátát (1e-3 a 3e-4 helyett) igényelnek. Ha a frissítés mértéke túl apró, a kerekítés egyszerűen elnyeli azt, így a súlyok beragadnának és a modell nem tudna tanulni.


**Koszinuszos tanulási ráta ütemező (Cosine LR Scheduler):** A MatMul-free modellnél a tanulási dinamika eltér a hagyományos transzformerektől, ezért **koszinuszos ütemezőt** alkalmaznak. A tanítás elején nagy lépések történnek (gyors konvergencia), a végén egyre kisebb lépések (finom hangolás). A tanulási ráta a koszinusz-függvény szerint csökken LEARNING_RATE-ről eta_min-re:

$$\eta_t = \eta_{\min} + \frac{1}{2}(\eta_0 - \eta_{\min})\left(1 + \cos\frac{\pi t}{T}\right)$$

ahol $t$ az aktuális lépés, $T$ az összes lépés száma, $\eta_0$ a kezdeti tanulási ráta.

In [ ]:
# Tanítási hiperparaméterek
BATCH_SIZE   = 16
LEARNING_RATE = 1e-3
MAX_ITERS    = 1000
EVAL_INTERVAL = 100
EVAL_ITERS   = 50

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-2)

print(f"Modell paraméterei: {sum(p.numel() for p in model.parameters()):,}")
print(f"Tanítási lépések:   {MAX_ITERS:,}")
print(f"Batch méret:        {BATCH_SIZE}")

# A tanulasi rata MAX_ITERS lepes alatt LEARNING_RATE-rol 1e-5-re csokken
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=MAX_ITERS,
    eta_min=1e-5
)

In [ ]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()

    for split, data in [('train', train_data), ('val', val_data)]:
        losses = []
        for _ in range(EVAL_ITERS):
            xb, yb = get_batch(data, BATCH_SIZE, config.seq_len, device)
            _, loss = model(xb, yb)
            losses.append(loss.item())
        out[split] = sum(losses) / len(losses)

    model.train()
    return out

In [ ]:
train_losses = []
val_losses   = []
log_steps    = []

print(f"Tanítás kezdete (eszköz: {device})")
print("-" * 50)

model.train()

for step in range(MAX_ITERS):

    if step % EVAL_INTERVAL == 0 or step == MAX_ITERS - 1:
        losses = estimate_loss()
        train_losses.append(losses['train'])
        val_losses.append(losses['val'])
        log_steps.append(step)
        print(f"Lépés {step:4d} | tanítási loss: {losses['train']:.4f} | validációs loss: {losses['val']:.4f}")

    xb, yb = get_batch(train_data, BATCH_SIZE, config.seq_len, device)
    logits, loss = model(xb, yb)
    loss.backward()
    optimizer.zero_grad()

    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

    optimizer.step()
    scheduler.step()

print("-" * 50)
print("Tanítás kész!")

Tanítás kezdete (eszköz: cpu)
--------------------------------------------------
Lépés    0 | tanítási loss: 5.5506 | validációs loss: 5.5509
Lépés  100 | tanítási loss: 5.5504 | validációs loss: 5.5512
Lépés  200 | tanítási loss: 5.5502 | validációs loss: 5.5509
Lépés  300 | tanítási loss: 5.5507 | validációs loss: 5.5509
Lépés  400 | tanítási loss: 5.5500 | validációs loss: 5.5504
Lépés  500 | tanítási loss: 5.5508 | validációs loss: 5.5503
Lépés  600 | tanítási loss: 5.5504 | validációs loss: 5.5509
Lépés  700 | tanítási loss: 5.5503 | validációs loss: 5.5509


Loss görbe

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(log_steps, train_losses, label='Tanítási loss', color='steelblue', linewidth=2, marker='o', markersize=4)
plt.plot(log_steps, val_losses,   label='Validációs loss', color='tomato',   linewidth=2, marker='s', markersize=4)

plt.xlabel('Tanítási lépés')
plt.ylabel('Cross-entropy loss')
plt.title('MatMul-Free LM – tanítási görbe')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Kezdeti loss:   {train_losses[0]:.4f}  (véletlenszerű jóslás esetén ~{math.log(config.vocab_size):.2f})")
print(f"Végső train:    {train_losses[-1]:.4f}")
print(f"Végső val:      {val_losses[-1]:.4f}")
print()
print(f"Kezdeti perplexity:  {math.exp(train_losses[0]):.1f}")
print(f"Végső perplexity:    {math.exp(train_losses[-1]):.1f}")
print(f"(A perplexity azt mutatja, hány egyformán valószínű lehetőség között 'habozik' a modell)")